In [2]:
"""
============================================================
  TDNN + CTC — Javanese ASR (Phoneme-level)
  dengan Hyperparameter Tuning Komprehensif
============================================================
  Pipeline:
    1. Ekstraksi MFCC 39-dim dari .wav
    2. G2P Bahasa Jawa → sequence fonem
    3. Hyperparameter Tuning (grid search lengkap)
    4. Training penuh model terbaik
    5. Evaluasi → WER + Loss + Akurasi Fonem

  Hyperparameter yang di-tuning:
    - Jumlah layer TDNN  : [3, 4, 5]
    - Hidden dim         : [128, 256, 512]
    - Learning rate      : [1e-3, 1e-4, 1e-5]
    - Dropout rate       : [0.1, 0.2, 0.3]
    - Context window     : [narrow, standard, wide]

  Output:
    HasilTuning_TDNN.csv   — ringkasan semua eksperimen
    HasilWER_TDNN.csv      — WER per audio (model terbaik)
    HasilDekode_TDNN.csv   — referensi vs hipotesis
    HasilTraining_TDNN.csv — history loss per epoch
    tdnn_best.npy          — bobot model terbaik
============================================================
"""

import os, re, time, warnings
import numpy as np
import pandas as pd
from itertools import product
from scipy.special import logsumexp

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────
#  KONFIGURASI PATH
# ─────────────────────────────────────────────────────────────

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/TDNN/out"
)
os.makedirs(DIR_SAVE, exist_ok=True)

OUT_TUNING_CSV  = os.path.join(DIR_SAVE, "HasilTuning_TDNN.csv")
OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN.csv")
MODEL_BEST_PATH = os.path.join(DIR_SAVE, "tdnn_best.npy")
NORM_PATH       = os.path.join(DIR_SAVE, "mfcc_global_norm.npy")

COL_PATH_DATA = "Path_Data"
COL_SENTENCES = "sentences"


# ─────────────────────────────────────────────────────────────
#  KONFIGURASI MFCC
# ─────────────────────────────────────────────────────────────

SR          = 16000
N_MFCC      = 13
N_FFT       = 512
HOP_LENGTH  = 160
WIN_LENGTH  = 400
N_MELS      = 26
PRE_EMPH    = 0.97
FMIN        = 0.0
FMAX        = 4000.0
DELTA_WIDTH = 2
INPUT_DIM   = N_MFCC * 3   # 39
FEAT_CLIP   = 10.0

BLANK = 0


# ─────────────────────────────────────────────────────────────
#  CONTEXT PRESETS PER JUMLAH LAYER
# ─────────────────────────────────────────────────────────────

CONTEXT_PRESETS = {
    3: {
        "narrow"  : [[-1, 1], [-1, 0], [0, 0]],
        "standard": [[-2, 2], [-2, 0], [0, 0]],
        "wide"    : [[-3, 3], [-3, 0], [0, 0]],
    },
    4: {
        "narrow"  : [[-1, 1], [-1, 0], [-2, 1], [0, 0]],
        "standard": [[-2, 2], [-2, 0], [-3, 3], [0, 0]],
        "wide"    : [[-3, 3], [-3, 0], [-5, 2], [0, 0]],
    },
    5: {
        "narrow"  : [[-1, 1], [-1, 0], [-2, 1], [-3, 1], [0, 0]],
        "standard": [[-2, 2], [-2, 0], [-3, 3], [-7, 2], [0, 0]],
        "wide"    : [[-3, 3], [-3, 0], [-5, 2], [-9, 3], [0, 0]],
    },
}


# ─────────────────────────────────────────────────────────────
#  TUNING GRID & FIXED CONFIG
# ─────────────────────────────────────────────────────────────

TUNING_GRID = {
    "n_layers"     : [3, 4, 5],
    "hidden_dim"   : [128, 256, 512],
    "learning_rate": [1e-3, 1e-4, 1e-5],
    "dropout"      : [0.1, 0.2, 0.3],
    "context_type" : ["narrow", "standard", "wide"],
}

FIXED_CONFIG = {
    "grad_clip"      : 1.0,
    "beam_width"     : 10,
    "epochs_tuning"  : 15,
    "epochs_full"    : 80,
    "early_stop"     : 5,
    "early_stop_full": 15,
    "lr_warmup"      : 1e-5,
    "warmup_epoch"   : 5,
    "lr_decay"       : 0.5,
    "lr_patience"    : 5,
    "lr_min"         : 1e-6,
    "train_ratio"    : 0.80,
    "val_ratio"      : 0.10,
    "batch_size"     : 16,
    "random_seed"    : 42,
}


# ─────────────────────────────────────────────────────────────
#  G2P BAHASA JAWA
#
#  DESAIN KUNCI:
#  Setiap fonem direpresentasikan sebagai TOKEN STRING ATOMIK,
#  bukan karakter tunggal Unicode. Ini mencegah ambiguitas saat
#  decode: 'NG' adalah 1 token, bukan 2 karakter 'N'+'G'.
#
#  Token fonem yang digunakan:
#    Huruf biasa : 'a','b','c','d','e','f','g','h','i','j',
#                  'k','l','m','n','o','p','q','r','s','t',
#                  'u','v','w','x','y','z'
#    Digraf      : 'NG', 'NY', 'TH', 'DH'  (huruf kapital)
#    Silence     : '<sp>'
#    Blank CTC   : '<blank>'  (index 0, tidak dari G2P)
# ─────────────────────────────────────────────────────────────

# Digraf → token atomik (huruf kapital agar tidak collision)
DIGRAPHS = {
    'ng': 'NG',
    'ny': 'NY',
    'th': 'TH',
    'dh': 'DH',
}

SILENCE_TOKEN = '<sp>'

_CHAR_MAP = {
    'é':'e','è':'e','ê':'e',
    'â':'a','à':'a','á':'a',
    'ó':'o','ô':'o',
    'í':'i','î':'i',
    'ú':'u','û':'u',
}

_ANGKA_JAWA = {
    0:'nol',1:'siji',2:'loro',3:'telu',4:'papat',
    5:'lima',6:'enem',7:'pitu',8:'wolu',9:'songo',
    10:'sepuluh',11:'sewelas',12:'rolas',
    20:'rongpuluh',100:'satus',1000:'sewu',
}

def _angka_ke_jawa(n: int) -> str:
    if n in _ANGKA_JAWA: return _ANGKA_JAWA[n]
    if n < 100:
        p = _ANGKA_JAWA.get(n//10*10, str(n//10)+'puluh')
        s = n % 10
        return p if s == 0 else p + _ANGKA_JAWA.get(s, str(s))
    if n < 1000:
        r = n//100; sisa = n % 100
        pfx = "satus" if r == 1 else str(r)+"atus"
        return pfx if sisa == 0 else pfx + _angka_ke_jawa(sisa)
    r = n//1000; sisa = n % 1000
    pfx = "sewu" if r == 1 else str(r)+"ewu"
    return pfx if sisa == 0 else pfx + _angka_ke_jawa(sisa)


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'\d+',
                  lambda m: ' ' + _angka_ke_jawa(int(m.group())) + ' ',
                  text)
    text = ''.join(_CHAR_MAP.get(c, c) for c in text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def javanese_g2p(text: str) -> list:
    """
    Teks Bahasa Jawa (sudah dinormalisasi) → list token fonem atomik.
    Return: list of str, contoh: ['s','a','NG','a','t','<sp>','b','a','i','k']
    """
    text     = normalize_text(text)
    phonemes = []
    i = 0
    while i < len(text):
        ch = text[i]
        if ch == ' ':
            if phonemes and phonemes[-1] != SILENCE_TOKEN:
                phonemes.append(SILENCE_TOKEN)
            i += 1
            continue
        if not ch.isalpha():
            i += 1
            continue
        # Cek digraf (2 karakter)
        if i + 1 < len(text) and text[i:i+2] in DIGRAPHS:
            phonemes.append(DIGRAPHS[text[i:i+2]])
            i += 2
            continue
        phonemes.append(ch)
        i += 1
    return phonemes


def build_phoneme_vocab(sentences: list) -> tuple:
    """
    Bangun vocab dari seluruh kalimat.
    Index 0 selalu '<blank>' (untuk CTC).
    """
    all_phn = set()
    for s in sentences:
        all_phn.update(javanese_g2p(s))
    vocab   = ['<blank>'] + sorted(all_phn)
    phn2id  = {p: i for i, p in enumerate(vocab)}
    id2phn  = {i: p for p, i in phn2id.items()}
    return vocab, phn2id, id2phn


def encode_phoneme(sentence: str, phn2id: dict) -> list:
    """Kalimat → list int ID fonem (tanpa blank)."""
    return [phn2id[p] for p in javanese_g2p(sentence) if p in phn2id]


def phoneme_tokens_to_text(tokens: list) -> str:
    """
    List token fonem atomik (str) → string teks.
    Input HARUS berupa list of str (token), bukan string.
    Contoh input : ['s','a','NG','a','t','<sp>','b','a','i','k']
    Contoh output: 'sangat baik'
    """
    parts = []
    for tok in tokens:
        if tok in (SILENCE_TOKEN, '<sp>'):
            parts.append(' ')
        elif tok == 'NG':
            parts.append('ng')
        elif tok == 'NY':
            parts.append('ny')
        elif tok == 'TH':
            parts.append('th')
        elif tok == 'DH':
            parts.append('dh')
        elif tok in ('<blank>', ''):
            pass
        else:
            parts.append(tok)
    return re.sub(r'\s+', ' ', ''.join(parts)).strip()


def tokens_to_words(tokens: list) -> list:
    """
    List token fonem → list kata (split pada silence).
    Contoh: ['s','a','NG','a','t','<sp>','b','a','i','k']
          → ['sangat', 'baik']
    """
    words   = []
    current = []
    for tok in tokens:
        if tok in (SILENCE_TOKEN, '<sp>'):
            if current:
                w = phoneme_tokens_to_text(current)
                if w.strip():
                    words.append(w.strip())
                current = []
        else:
            current.append(tok)
    if current:
        w = phoneme_tokens_to_text(current)
        if w.strip():
            words.append(w.strip())
    return words


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC
# ─────────────────────────────────────────────────────────────

try:
    import librosa; LIBROSA_OK = True
except ImportError:
    LIBROSA_OK = False
try:
    from scipy.io import wavfile; SCIPY_OK = True
except ImportError:
    SCIPY_OK = False


def extract_features_raw(wav_path: str) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(wav_path)
    return _librosa_raw(wav_path) if LIBROSA_OK else _manual_raw(wav_path)


def _librosa_raw(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0: raise ValueError("Audio kosong")
    y    = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])
    mfcc = librosa.feature.mfcc(
        y=y, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX, window="hamming"
    )
    d1 = librosa.feature.delta(mfcc, width=DELTA_WIDTH*2+1)
    d2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH*2+1)
    return np.vstack([mfcc, d1, d2]).T.astype(np.float32)


def _delta_vec(feat, width=2):
    T     = feat.shape[0]
    denom = 2.0 * sum(n*n for n in range(1, width+1))
    delta = np.zeros_like(feat)
    for n in range(1, width+1):
        delta += n * (feat[np.clip(np.arange(T)+n, 0, T-1)] -
                      feat[np.clip(np.arange(T)-n, 0, T-1)])
    return delta / denom


def _manual_raw(wav_path):
    if not SCIPY_OK: raise ImportError("Butuh librosa atau scipy")
    sr, audio = wavfile.read(wav_path)
    if audio.dtype == np.int16:   audio = audio / 32768.0
    elif audio.dtype == np.int32: audio = audio / 2147483648.0
    audio = audio.astype(np.float32)
    if sr != SR:
        n     = int(len(audio) * SR / sr)
        audio = np.interp(np.linspace(0, len(audio)-1, n),
                          np.arange(len(audio)), audio)
    y     = np.append(audio[0], audio[1:] - PRE_EMPH * audio[:-1])
    T_fr  = 1 + (len(y) - WIN_LENGTH) // HOP_LENGTH
    idx   = np.arange(WIN_LENGTH)[None,:] + np.arange(T_fr)[:,None] * HOP_LENGTH
    frames = y[idx] * np.hamming(WIN_LENGTH)
    fft_m  = np.abs(np.fft.rfft(frames, n=N_FFT))
    def hz2mel(f): return 1127 * np.log1p(f / 700)
    def mel2hz(m): return 700 * (np.exp(m / 1127) - 1)
    pts  = mel2hz(np.linspace(hz2mel(FMIN), hz2mel(FMAX), N_MELS+2))
    bins = np.floor((N_FFT+1) * pts / SR).astype(int)
    fb   = np.zeros((N_MELS, N_FFT//2+1))
    for m in range(1, N_MELS+1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1,k] = (k - bins[m-1]) / (bins[m] - bins[m-1] + 1e-10)
        for k in range(bins[m], bins[m+1]):
            fb[m-1,k] = (bins[m+1] - k) / (bins[m+1] - bins[m] + 1e-10)
    log_mel = np.log10(np.maximum(fft_m**2 @ fb.T, 1e-10))
    N_  = N_MELS
    ni  = np.arange(N_MFCC)[:,None]; ki = np.arange(N_)[None,:] + 0.5
    mfcc = log_mel @ (np.cos(np.pi * ni * ki / N_) * np.sqrt(2/N_)).T
    d1   = _delta_vec(mfcc, DELTA_WIDTH); d2 = _delta_vec(d1, DELTA_WIDTH)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  NORMALISASI MFCC GLOBAL
# ─────────────────────────────────────────────────────────────

def compute_global_norm(sequences: dict, train_ids: list) -> tuple:
    print("  Menghitung normalisasi global dari train set ...")
    all_frames = np.concatenate([sequences[a] for a in train_ids], axis=0)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std  = (all_frames.std(axis=0) + 1e-8).astype(np.float32)
    return mean, std


def apply_global_norm(sequences: dict, mean: np.ndarray,
                      std: np.ndarray) -> dict:
    return {
        aid: np.clip((feat - mean) / std, -FEAT_CLIP, FEAT_CLIP).astype(np.float32)
        for aid, feat in sequences.items()
    }


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if (os.path.splitext(f)[0] == audio_id and
                    f.lower().endswith(".wav")):
                return os.path.join(root, f)
    return ""


def load_data(csv_path: str) -> tuple:
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")

    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ada.")

    sentences_norm = [normalize_text(str(s)) for s in df[COL_SENTENCES].tolist()]
    vocab, phn2id, id2phn = build_phoneme_vocab(sentences_norm)
    print(f"  Vocab fonem : {len(vocab)-1} fonem + 1 blank = {len(vocab)} unit")
    print(f"  Fonem       : {sorted(set(vocab) - {'<blank>'})}")

    # Verifikasi: semua token fonem harus string (bukan karakter tunggal saja)
    sample_sent  = sentences_norm[0] if sentences_norm else ""
    sample_toks  = javanese_g2p(sample_sent)
    print(f"  Contoh G2P  : '{sample_sent[:40]}' → {sample_toks[:10]}")

    sequences_raw = {}; targets = {}
    word_refs = {};     phn_refs = {}
    skipped   = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(
            os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = normalize_text(str(row[COL_SENTENCES]))
        words    = kalimat.split()

        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)
        if not wav_path:
            skipped += 1; continue

        try:
            feats  = extract_features_raw(wav_path)
            T      = feats.shape[0]
            target = encode_phoneme(kalimat, phn2id)
            if T == 0 or not target:
                skipped += 1; continue
            if T < 2 * len(target) + 1:
                skipped += 1; continue
            sequences_raw[audio_id] = feats
            targets[audio_id]       = target
            word_refs[audio_id]     = words
            # Simpan referensi fonem sebagai list TOKEN (str), bukan ID
            phn_refs[audio_id]      = [id2phn[i] for i in target]
        except Exception as e:
            print(f"  ⚠ {audio_id}: {e}")
            skipped += 1; continue

        if (idx+1) % 200 == 0 or (idx+1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    actual_dim = next(iter(sequences_raw.values())).shape[1]
    lens = [len(targets[a]) for a in sequences_raw]
    print(f"\n  Audio berhasil : {len(sequences_raw)} | dilewati: {skipped}")
    print(f"  Panjang fonem  : min={min(lens)}, "
          f"mean={np.mean(lens):.1f}, max={max(lens)}")
    return (sequences_raw, targets, word_refs, phn_refs,
            phn2id, id2phn, vocab, actual_dim)


# ─────────────────────────────────────────────────────────────
#  CTC LOSS + GRADIENT (log-space, vectorized)
# ─────────────────────────────────────────────────────────────

def _softmax(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)


def ctc_loss_and_grad(logits: np.ndarray, target_ids: list) -> tuple:
    T, V = logits.shape
    L    = len(target_ids)
    S    = 2 * L + 1
    NEG  = -1e9

    l_ = np.zeros(S, dtype=np.int32)
    for i, c in enumerate(target_ids):
        l_[2*i+1] = c

    probs     = _softmax(logits).astype(np.float64)
    log_probs = np.log(probs + 1e-30)
    lp_ext    = log_probs[:, l_]

    can_skip = np.zeros(S, dtype=bool)
    for s in range(2, S):
        if l_[s] != BLANK and l_[s] != l_[s-2]:
            can_skip[s] = True
    can_skip_bk = np.zeros(S, dtype=bool)
    for s in range(S-2):
        if l_[s+2] != BLANK and l_[s+2] != l_[s]:
            can_skip_bk[s] = True

    alpha = np.full((T, S), NEG)
    alpha[0, 0] = lp_ext[0, 0]
    if S > 1: alpha[0, 1] = lp_ext[0, 1]
    for t in range(1, T):
        s0 = alpha[t-1]
        s1 = np.empty(S); s1[0] = NEG; s1[1:] = alpha[t-1, :-1]
        s2 = np.empty(S); s2[:2] = NEG; s2[2:] = alpha[t-1, :-2]
        base     = np.logaddexp(s0, s1)
        alpha[t] = np.where(can_skip, np.logaddexp(base, s2), base) + lp_ext[t]

    beta = np.full((T, S), NEG)
    beta[T-1, S-1] = 0.0
    if S > 1: beta[T-1, S-2] = 0.0
    for t in range(T-2, -1, -1):
        lp  = lp_ext[t+1]
        s0  = beta[t+1] + lp
        s1  = np.empty(S); s1[-1] = NEG; s1[:-1] = beta[t+1, 1:] + lp[1:]
        s2  = np.empty(S); s2[-2:] = NEG; s2[:-2] = beta[t+1, 2:] + lp[2:]
        base    = np.logaddexp(s0, s1)
        beta[t] = np.where(can_skip_bk, np.logaddexp(base, s2), base)

    ll_terms = [alpha[T-1, S-1]]
    if S > 1: ll_terms.append(alpha[T-1, S-2])
    ll = float(logsumexp(ll_terms))
    if not np.isfinite(ll) or ll < -1e7:
        return 0.0, np.zeros((T, V), dtype=np.float32)

    loss      = float(-ll / T)
    ab        = alpha + beta
    log_numer = np.full((T, V), NEG)
    for s in range(S):
        log_numer[:, l_[s]] = np.logaddexp(log_numer[:, l_[s]], ab[:, s])
    term = np.exp(np.clip(log_numer - ll, -500, 0))
    return loss, (probs - term).astype(np.float32)


# ─────────────────────────────────────────────────────────────
#  LAYER NORM
# ─────────────────────────────────────────────────────────────

class LayerNorm:
    def __init__(self, dim, lr=1e-3, eps=1e-5):
        self.eps = eps; self.lr = lr
        self.gamma = np.ones(dim,  dtype=np.float32)
        self.beta  = np.zeros(dim, dtype=np.float32)
        self.mg    = np.zeros(dim, dtype=np.float32)
        self.vg    = np.zeros(dim, dtype=np.float32)
        self.mb    = np.zeros(dim, dtype=np.float32)
        self.vb    = np.zeros(dim, dtype=np.float32)
        self.t     = 0
        self._xhat = None; self._std = None
        self._dg_acc = np.zeros(dim, dtype=np.float32)
        self._db_acc = np.zeros(dim, dtype=np.float32)
        self._n_acc  = 0

    def forward(self, x):
        mean = x.mean(axis=1, keepdims=True)
        std  = np.sqrt(x.var(axis=1, keepdims=True) + self.eps)
        xhat = (x - mean) / std
        self._xhat = xhat; self._std = std
        return self.gamma * xhat + self.beta

    def backward(self, dout, accumulate=True):
        T, D = dout.shape
        dg   = (dout * self._xhat).sum(axis=0)
        db   = dout.sum(axis=0)
        dxhat = dout * self.gamma
        dx = (1.0 / D / self._std) * (
            D * dxhat
            - dxhat.sum(axis=1, keepdims=True)
            - self._xhat * (dxhat * self._xhat).sum(axis=1, keepdims=True)
        )
        if accumulate:
            self._dg_acc += dg; self._db_acc += db; self._n_acc += 1
        else:
            self._adam(dg, db)
        return dx

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dg_acc / self._n_acc, self._db_acc / self._n_acc)
        self._dg_acc[:] = 0; self._db_acc[:] = 0; self._n_acc = 0

    def set_lr(self, lr): self.lr = lr

    def _adam(self, dg, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mg = b1*self.mg + (1-b1)*dg
        self.vg = b2*self.vg + (1-b2)*dg**2
        self.gamma -= self.lr * (self.mg/(1-b1**self.t)) / (
            np.sqrt(self.vg/(1-b2**self.t)) + eps)
        self.mb = b1*self.mb + (1-b1)*db
        self.vb = b2*self.vb + (1-b2)*db**2
        self.beta -= self.lr * (self.mb/(1-b1**self.t)) / (
            np.sqrt(self.vb/(1-b2**self.t)) + eps)


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim, out_dim, context, lr, use_ln=True):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr
        self.use_ln  = use_ln
        fan_in = in_dim * self.ctx_len
        self.W = (np.random.randn(fan_in, out_dim) *
                  np.sqrt(2.0 / fan_in)).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)
        if use_ln: self.ln = LayerNorm(out_dim, lr=lr)
        self.mW = np.zeros_like(self.W); self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b); self.vb = np.zeros_like(self.b)
        self.t  = 0; self._X = None
        self._dW_acc = np.zeros_like(self.W)
        self._db_acc = np.zeros_like(self.b)
        self._n_acc  = 0

    def splice(self, X):
        T   = X.shape[0]
        idx = np.arange(T)
        return np.concatenate(
            [X[np.clip(idx + off, 0, T-1)]
             for off in range(self.context[0], self.context[1]+1)],
            axis=1
        )

    def forward(self, Xsp):
        self._X = Xsp
        z = Xsp @ self.W + self.b
        if self.use_ln: z = self.ln.forward(z)
        return z

    def backward(self, dout, accumulate=True):
        T = len(dout)
        if self.use_ln: dout = self.ln.backward(dout, accumulate=accumulate)
        dW   = self._X.T @ dout / T
        db   = dout.mean(0)
        dXsp = dout @ self.W.T
        if accumulate:
            self._dW_acc += dW; self._db_acc += db; self._n_acc += 1
        else:
            self._adam(dW, db)
        B, D_tot = dXsp.shape
        D = D_tot // self.ctx_len
        dX = np.zeros((B, D), dtype=np.float32)
        for k in range(self.ctx_len):
            dX += dXsp[:, k*D:(k+1)*D]
        return dX

    def flush(self):
        if self._n_acc == 0: return
        self._adam(self._dW_acc / self._n_acc, self._db_acc / self._n_acc)
        if self.use_ln: self.ln.flush()
        self._dW_acc[:] = 0; self._db_acc[:] = 0; self._n_acc = 0

    def set_lr(self, lr):
        self.lr = lr
        if self.use_ln: self.ln.set_lr(lr)

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        self.mW = b1*self.mW + (1-b1)*dW
        self.vW = b2*self.vW + (1-b2)*dW**2
        self.W -= self.lr * (self.mW/(1-b1**self.t)) / (
            np.sqrt(self.vW/(1-b2**self.t)) + eps)
        self.mb = b1*self.mb + (1-b1)*db
        self.vb = b2*self.vb + (1-b2)*db**2
        self.b -= self.lr * (self.mb/(1-b1**self.t)) / (
            np.sqrt(self.vb/(1-b2**self.t)) + eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dim, contexts,
                 output_dim, dropout=0.2, lr=1e-3):
        """
        contexts : list of [left, right] — satu entry per hidden layer.
                   Output layer [0,0] ditambah otomatis.
        """
        self.dropout  = dropout
        self.training = True
        self.layers   = []
        in_d = input_dim
        for ctx in contexts:
            self.layers.append(
                TDNNLayer(in_d, hidden_dim, ctx, lr, use_ln=True))
            in_d = hidden_dim
        # Output layer — tanpa LayerNorm
        self.layers.append(
            TDNNLayer(in_d, output_dim, [0, 0], lr, use_ln=False))
        self._rc = []; self._dc = []

    def set_training(self, mode): self.training = mode

    def set_lr(self, lr):
        for l in self.layers: l.set_lr(lr)

    def forward(self, X):
        self._rc = []; self._dc = []
        h = X
        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = np.maximum(0.0, z)
            if self.training and self.dropout > 0:
                mask = (np.random.rand(*a.shape) > self.dropout
                        ).astype(np.float32)
                a    = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
            self._rc.append(z); self._dc.append(mask)
            h = a
        out = self.layers[-1]
        return out.forward(out.splice(h))

    def ctc_step(self, X, target, grad_clip=1.0, accumulate=True):
        logits = self.forward(X)
        loss, dlogit = ctc_loss_and_grad(logits, target)
        if loss == 0.0: return 0.0
        gnorm = np.sqrt((dlogit**2).sum())
        if gnorm > grad_clip: dlogit = dlogit * (grad_clip / gnorm)
        sc = 1.0 / (1.0 - self.dropout + 1e-9) if self.dropout > 0 else 1.0
        dh = self.layers[-1].backward(dlogit, accumulate=accumulate)
        for i in reversed(range(len(self.layers) - 1)):
            dh = dh * self._dc[i] * sc
            dh = dh * (self._rc[i] > 0).astype(np.float32)
            dh = self.layers[i].backward(dh, accumulate=accumulate)
        return loss

    def flush(self):
        for l in self.layers: l.flush()

    def get_log_probs(self, X):
        self.set_training(False)
        p = _softmax(self.forward(X))
        return np.log(p + 1e-30)

    def save(self, path):
        np.save(path, {
            "w" : [(l.W.copy(), l.b.copy()) for l in self.layers],
            "ln": [(l.ln.gamma.copy(), l.ln.beta.copy()) if l.use_ln
                   else (None, None) for l in self.layers],
        }, allow_pickle=True)

    def load(self, path):
        d = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(d["w"]):
            self.layers[i].W = W; self.layers[i].b = b
        if "ln" in d:
            for i, (g, bt) in enumerate(d["ln"]):
                if g is not None and self.layers[i].use_ln:
                    self.layers[i].ln.gamma = g
                    self.layers[i].ln.beta  = bt
        print(f"  Model dimuat: {path}")


# ─────────────────────────────────────────────────────────────
#  SPLIT DATASET
# ─────────────────────────────────────────────────────────────

def split_dataset(ids, cfg):
    rng  = np.random.default_rng(cfg["random_seed"])
    ids  = list(ids)
    perm = rng.permutation(len(ids))
    n    = len(ids)
    ntr  = int(n * cfg["train_ratio"])
    nva  = int(n * cfg["val_ratio"])
    tr   = [ids[i] for i in perm[:ntr]]
    va   = [ids[i] for i in perm[ntr:ntr+nva]]
    te   = [ids[i] for i in perm[ntr+nva:]]
    return tr, va, te


# ─────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────

def get_lr(epoch, no_improve, cfg, lr_init):
    if epoch <= cfg["warmup_epoch"]:
        alpha = (epoch - 1) / max(cfg["warmup_epoch"] - 1, 1)
        return cfg["lr_warmup"] + alpha * (lr_init - cfg["lr_warmup"])
    return max(
        lr_init * (cfg["lr_decay"] ** (no_improve // cfg["lr_patience"])),
        cfg["lr_min"]
    )


# ─────────────────────────────────────────────────────────────
#  TRAINING LOOP
# ─────────────────────────────────────────────────────────────

def run_training(model, train_ids, val_ids, sequences, targets,
                 cfg, lr_init, max_epochs, early_stop,
                 model_save_path=None, verbose=True):
    rng       = np.random.default_rng(cfg["random_seed"])
    hist      = []; best = float("inf"); pat = 0; no_improve = 0
    grad_clip = cfg["grad_clip"]; batch_size = cfg["batch_size"]

    if verbose:
        print(f"\n  {'Ep':>4}  {'LR':>8}  {'TrLoss':>8}  "
              f"{'VaLoss':>8}  {'Blank%':>7}  {'t':>6}")
        print(f"  {'-'*52}")

    for epoch in range(1, max_epochs+1):
        t0  = time.time()
        lr  = get_lr(epoch, no_improve, cfg, lr_init)
        model.set_lr(lr); model.set_training(True)

        perm  = rng.permutation(len(train_ids))
        order = [train_ids[i] for i in perm]
        total_loss = 0.0; n_batches = 0

        for start in range(0, len(order), batch_size):
            batch = order[start:start+batch_size]
            bl = 0.0; valid = 0
            for aid in batch:
                try:
                    lv = model.ctc_step(sequences[aid], targets[aid],
                                        grad_clip=grad_clip, accumulate=True)
                    if lv > 0: bl += lv; valid += 1
                except Exception:
                    continue
            model.flush()
            if valid > 0: total_loss += bl / valid; n_batches += 1

        tr_loss = total_loss / max(n_batches, 1)

        model.set_training(False)
        vl = 0.0; vn = 0
        for aid in val_ids:
            try:
                logits = model.forward(sequences[aid])
                lv, _  = ctc_loss_and_grad(logits, targets[aid])
                if lv > 0: vl += lv; vn += 1
            except Exception:
                continue
        val_loss = vl / max(vn, 1)

        blank_pct = _blank_pct(model, val_ids[:15], sequences)
        elapsed   = time.time() - t0
        hist.append({
            "epoch"     : epoch,
            "lr"        : round(lr, 8),
            "train_loss": round(tr_loss, 6),
            "val_loss"  : round(val_loss, 6),
            "blank_pct" : round(blank_pct, 1),
        })

        if verbose:
            print(f"  {epoch:4d}  {lr:8.6f}  {tr_loss:8.4f}  "
                  f"{val_loss:8.4f}  {blank_pct:6.1f}%  {elapsed:5.0f}s")
            if blank_pct > 97.0 and epoch > cfg["warmup_epoch"]:
                print(f"  ⚠ Blank collapse ({blank_pct:.1f}%)")

        if val_loss < best - 1e-5:
            best = val_loss; pat = 0; no_improve = 0
            if model_save_path: model.save(model_save_path)
        else:
            pat += 1; no_improve += 1
            if pat >= early_stop:
                if verbose:
                    print(f"\n  ✓ Early stop ep {epoch} (best={best:.4f})")
                break

    return best, hist


def _blank_pct(model, ids, sequences, n=10):
    model.set_training(False)
    tb = tf = 0
    for aid in ids[:n]:
        lp    = model.get_log_probs(sequences[aid])
        preds = np.argmax(lp, axis=1)
        tb   += (preds == BLANK).sum()
        tf   += len(preds)
    return tb / max(tf, 1) * 100


# ─────────────────────────────────────────────────────────────
#  CTC BEAM SEARCH DECODE
#
#  Return: list of str (token fonem atomik)
#  Contoh: ['s','a','NG','a','t','<sp>','b','a','i','k']
#
#  CATATAN PENTING:
#  Fungsi ini mengembalikan LIST TOKEN, bukan string.
#  Konversi ke string dilakukan oleh phoneme_tokens_to_text().
#  Konversi ke kata dilakukan oleh tokens_to_words().
#  Ini mencegah ambiguitas digraf saat evaluasi akurasi fonem.
# ─────────────────────────────────────────────────────────────

def ctc_beam_decode_tokens(log_probs: np.ndarray,
                           id2phn: dict,
                           beam_width: int = 10) -> list:
    """CTC Beam Search → list token fonem atomik (str)."""
    T, V = log_probs.shape
    NEG  = -float('inf')
    beam = {(): (0.0, NEG)}   # { seq: (log_pb, log_pnb) }

    for t in range(T):
        new_beam = {}
        for seq, (pb, pnb) in beam.items():
            # Blank extension
            lp_b      = log_probs[t, BLANK]
            p0, p1    = new_beam.get(seq, (NEG, NEG))
            new_beam[seq] = (np.logaddexp(p0,
                                          np.logaddexp(pb, pnb) + lp_b), p1)
            # Non-blank extension
            for c in range(1, V):
                lp_c = log_probs[t, c]
                nseq = seq + (c,)
                # Repeat token hanya bisa via blank
                if seq and seq[-1] == c:
                    ep_nb = pb + lp_c
                else:
                    ep_nb = np.logaddexp(pb, pnb) + lp_c
                p0n, p1n = new_beam.get(nseq, (NEG, NEG))
                new_beam[nseq] = (p0n, np.logaddexp(p1n, ep_nb))

        beam = dict(
            sorted(new_beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]),
                   reverse=True)[:beam_width]
        )

    best_seq = max(beam.items(),
                   key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]))[0]

    # Konversi ID → token fonem atomik; buang blank
    tokens = [
        id2phn[int(c)]
        for c in best_seq
        if id2phn.get(int(c), '<blank>') != '<blank>'
    ]
    return tokens


# ─────────────────────────────────────────────────────────────
#  AKURASI FONEM  (Phone Error Rate & Accuracy)
#
#  Perbandingan di level TOKEN fonem atomik (str),
#  bukan di level karakter string.
#
#  PER  = edit_distance(ref_tokens, hyp_tokens) / len(ref_tokens)
#  Acc  = max(0, 1 - PER) × 100
# ─────────────────────────────────────────────────────────────

def compute_phoneme_accuracy(model, eval_ids, sequences, phn_refs,
                              id2phn, beam_width=10):
    """
    phn_refs : dict { audio_id: list_of_phoneme_tokens_str }
    Return   : (accuracy_pct, per_pct)
    """
    model.set_training(False)
    total_ref  = 0
    total_edit = 0

    for aid in eval_ids:
        lp       = model.get_log_probs(sequences[aid])
        hyp_toks = ctc_beam_decode_tokens(lp, id2phn, beam_width)
        ref_toks = phn_refs.get(aid, [])

        # Buang silence token — evaluasi fonem murni
        ref_clean = [t for t in ref_toks  if t not in (SILENCE_TOKEN, '<sp>')]
        hyp_clean = [t for t in hyp_toks  if t not in (SILENCE_TOKEN, '<sp>')]

        R = len(ref_clean)
        if R == 0: continue
        H = len(hyp_clean)

        # Levenshtein token-level
        dp = np.zeros((R+1, H+1), dtype=np.int32)
        dp[:, 0] = np.arange(R+1)
        dp[0, :] = np.arange(H+1)
        for i in range(1, R+1):
            for j in range(1, H+1):
                cost = 0 if ref_clean[i-1] == hyp_clean[j-1] else 1
                dp[i, j] = min(dp[i-1, j] + 1,
                               dp[i, j-1] + 1,
                               dp[i-1, j-1] + cost)
        total_ref  += R
        total_edit += int(dp[R, H])

    if total_ref == 0:
        return 0.0, 100.0
    per = total_edit / total_ref * 100.0
    acc = max(0.0, 100.0 - per)
    return round(acc, 4), round(per, 4)


# ─────────────────────────────────────────────────────────────
#  WER  (Word Error Rate)
# ─────────────────────────────────────────────────────────────

def levenshtein_words(ref: list, hyp: list) -> tuple:
    """Edit distance kata-level. Return (wer_float, S, D, I, N)."""
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R+1, H+1), dtype=np.int32)
    dp[:, 0] = np.arange(R+1)
    dp[0, :] = np.arange(H+1)
    for i in range(1, R+1):
        for j in range(1, H+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i, j] = min(dp[i-1, j]+1, dp[i, j-1]+1, dp[i-1, j-1]+cost)
    i, j = R, H; S = D = I = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            if dp[i, j] == dp[i-1, j-1] + cost:
                if cost: S += 1
                i -= 1; j -= 1
            elif dp[i, j] == dp[i-1, j] + 1:
                D += 1; i -= 1
            else:
                I += 1; j -= 1
        elif i > 0:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    N   = max(R, 1)
    wer = (S + D + I) / N   # tidak di-cap — WER bisa > 100%
    return wer, S, D, I, N


def evaluate_wer(model, eval_ids, sequences, word_refs, phn_refs,
                 id2phn, beam_width=10):
    """Satu decode per audio, hasilkan WER rows + summary."""
    model.set_training(False)
    wer_rows = []; tS = tD = tI = tN = 0

    for aid in eval_ids:
        lp        = model.get_log_probs(sequences[aid])
        hyp_toks  = ctc_beam_decode_tokens(lp, id2phn, beam_width)
        hyp_words = tokens_to_words(hyp_toks)
        hyp_text  = phoneme_tokens_to_text(hyp_toks)
        ref_words = word_refs.get(aid, [])

        wer_val, S, D, I, N = levenshtein_words(ref_words, hyp_words)
        tS += S; tD += D; tI += I; tN += N

        wer_rows.append({
            "audio_id" : aid,
            "WER"      : round(wer_val * 100, 4),
            "S"        : S, "D": D, "I": I, "N": N,
            "referensi": " ".join(ref_words),
            "hipotesis": hyp_text,
        })

    wer_corpus = (tS + tD + tI) / max(tN, 1) * 100
    wl         = [r["WER"] for r in wer_rows]
    summary = {
        "wer_corpus": round(wer_corpus, 4),
        "wer_mean"  : round(float(np.mean(wl)), 4),
        "wer_min"   : round(float(np.min(wl)), 4),
        "wer_max"   : round(float(np.max(wl)), 4),
        "total_S"   : tS, "total_D": tD, "total_I": tI, "total_N": tN,
        "n_audio"   : len(eval_ids),
    }
    return wer_rows, summary


def print_eval_stats(summ_wer, phn_acc_pct, phn_per_pct, split_name="test"):
    print(f"\n── Evaluasi [{split_name}] ────────────────────────────────")
    print(f"  Jumlah audio     : {summ_wer['n_audio']}")
    print(f"  WER Corpus       : {summ_wer['wer_corpus']:.2f}%")
    print(f"  WER Mean         : {summ_wer['wer_mean']:.2f}%")
    print(f"  WER Min / Max    : {summ_wer['wer_min']:.2f}% / "
          f"{summ_wer['wer_max']:.2f}%")
    print(f"  Akurasi Fonem    : {phn_acc_pct:.2f}%")
    print(f"  PER (Phone Err)  : {phn_per_pct:.2f}%")
    print(f"  Total S/D/I/N    : {summ_wer['total_S']}/"
          f"{summ_wer['total_D']}/{summ_wer['total_I']}/"
          f"{summ_wer['total_N']}")
    print(f"────────────────────────────────────────────────────────")


# ─────────────────────────────────────────────────────────────
#  HYPERPARAMETER TUNING — GRID SEARCH
# ─────────────────────────────────────────────────────────────

def _quick_eval(model, sample_ids, sequences, word_refs,
                phn_refs, id2phn, beam_width):
    """Evaluasi cepat satu kali decode untuk tuning."""
    _, wer_s    = evaluate_wer(model, sample_ids, sequences,
                                word_refs, phn_refs, id2phn, beam_width)
    phn_acc, phn_per = compute_phoneme_accuracy(
        model, sample_ids, sequences, phn_refs, id2phn, beam_width)
    return wer_s["wer_corpus"], phn_acc, phn_per


def hyperparameter_tuning(sequences, targets, word_refs, phn_refs,
                          vocab, id2phn, actual_dim,
                          train_ids, val_ids, cfg):
    param_names  = list(TUNING_GRID.keys())
    param_values = list(TUNING_GRID.values())
    combos       = list(product(*param_values))
    n_total      = len(combos)

    print(f"\n{'='*70}")
    print(f"  HYPERPARAMETER TUNING — GRID SEARCH ({n_total} eksperimen)")
    print(f"{'='*70}")
    for k, v in TUNING_GRID.items():
        print(f"  {k:<20}: {v}")
    print(f"\n  Epoch per eksperimen : {cfg['epochs_tuning']}")
    print(f"  Early stop           : {cfg['early_stop']}")
    print(f"  Metrik               : val_loss ↓ | WER% ↓ | AkFonem% ↑")
    print(f"{'='*70}")

    # Sample val tetap untuk semua eksperimen (konsistensi)
    rng_s      = np.random.default_rng(cfg["random_seed"])
    sample_ids = list(rng_s.choice(val_ids, min(50, len(val_ids)), replace=False))

    results       = []
    best_val_loss = float("inf")
    best_combo    = None

    for exp_idx, combo in enumerate(combos, 1):
        params   = dict(zip(param_names, combo))
        n_layers = params["n_layers"]
        h_dim    = params["hidden_dim"]
        lr       = params["learning_rate"]
        drop     = params["dropout"]
        ctx_type = params["context_type"]
        contexts = CONTEXT_PRESETS[n_layers][ctx_type]

        print(f"\n[{exp_idx:>4}/{n_total}] "
              f"n_layers={n_layers}  hidden_dim={h_dim}  "
              f"lr={lr:.0e}  dropout={drop}  context={ctx_type}")

        np.random.seed(cfg["random_seed"])
        model = TDNNModel(
            input_dim  = actual_dim,
            hidden_dim = h_dim,
            contexts   = contexts,
            output_dim = len(vocab),
            dropout    = drop,
            lr         = cfg["lr_warmup"],
        )

        best_vl, _ = run_training(
            model, train_ids, val_ids, sequences, targets,
            cfg, lr_init=lr,
            max_epochs=cfg["epochs_tuning"],
            early_stop=cfg["early_stop"],
            model_save_path=None,
            verbose=True,
        )

        wer_val, phn_acc, phn_per = _quick_eval(
            model, sample_ids, sequences,
            word_refs, phn_refs, id2phn, cfg["beam_width"])

        row = {
            "eksperimen"      : exp_idx,
            "n_layers"        : n_layers,
            "hidden_dim"      : h_dim,
            "learning_rate"   : lr,
            "dropout"         : drop,
            "context_type"    : ctx_type,
            "context_detail"  : str(contexts),
            "best_val_loss"   : round(best_vl, 6),
            "wer_val_pct"     : round(wer_val, 2),
            "phoneme_acc_val" : round(phn_acc, 2),
            "per_val"         : round(phn_per, 2),
        }
        results.append(row)
        print(f"  → val_loss={best_vl:.4f}  WER={wer_val:.1f}%  "
              f"PhAcc={phn_acc:.1f}%  PER={phn_per:.1f}%")

        if best_vl < best_val_loss:
            best_val_loss = best_vl
            best_combo    = params.copy()
            best_combo["contexts"] = contexts

    df_tune = (pd.DataFrame(results)
               .sort_values(["best_val_loss", "wer_val_pct"])
               .reset_index(drop=True))
    df_tune.to_csv(OUT_TUNING_CSV, index=False)
    print(f"\n  HasilTuning_TDNN.csv tersimpan ({n_total} eksperimen)")
    _print_tuning_table(df_tune)
    print(f"\n  Config terbaik → {best_combo}")
    return best_combo, df_tune


def _print_tuning_table(df):
    best_loss = df["best_val_loss"].min()
    print(f"\n{'='*84}")
    print(f"  RINGKASAN HASIL TUNING (20 terbaik berdasarkan val_loss)")
    print(f"{'='*84}")
    print(f"  {'Eks':>4}  {'nL':>3}  {'hDim':>5}  {'LR':>7}  {'Drop':>5}  "
          f"{'Ctx':>8}  {'ValLoss':>8}  {'WER%':>7}  {'PhAcc%':>7}  {'PER%':>6}")
    print(f"  {'-'*80}")
    for _, r in df.head(20).iterrows():
        mark = " ←" if r["best_val_loss"] == best_loss else ""
        print(f"  {int(r['eksperimen']):4d}  "
              f"{int(r['n_layers']):3d}  "
              f"{int(r['hidden_dim']):5d}  "
              f"{r['learning_rate']:7.1e}  "
              f"{r['dropout']:5.1f}  "
              f"{r['context_type']:>8}  "
              f"{r['best_val_loss']:8.4f}  "
              f"{r['wer_val_pct']:6.1f}%  "
              f"{r['phoneme_acc_val']:6.1f}%  "
              f"{r['per_val']:5.1f}%{mark}")
    if len(df) > 20:
        print(f"  ... (lihat CSV untuk {len(df)} eksperimen lengkap)")
    print(f"{'='*84}")


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(FIXED_CONFIG["random_seed"])

    print("="*70)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC")
    print("="*70)
    print(f"  Folder audio : {SAMPLED_ROOT}")
    print(f"  CSV          : {CSV_PATH}")
    print(f"  Output       : {DIR_SAVE}")

    (sequences_raw, targets, word_refs, phn_refs,
     phn2id, id2phn, vocab, actual_dim) = load_data(CSV_PATH)

    all_ids = list(sequences_raw.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids, FIXED_CONFIG)
    print(f"\n  Split → train:{len(train_ids)}  val:{len(val_ids)}  "
          f"test:{len(test_ids)}")

    mean_g, std_g = compute_global_norm(sequences_raw, train_ids)
    np.save(NORM_PATH, {"mean": mean_g, "std": std_g})
    sequences = apply_global_norm(sequences_raw, mean_g, std_g)
    print(f"  MFCC normalisasi + clip [-{FEAT_CLIP},{FEAT_CLIP}] selesai")

    # ── TAHAP 2 ──────────────────────────────────────────────
    print("\n" + "="*70)
    print("  TAHAP 2 — Hyperparameter Tuning (Grid Search)")
    print("="*70)
    n_combos = 1
    for v in TUNING_GRID.values(): n_combos *= len(v)
    print(f"  Total kombinasi : {n_combos}")

    best_params, df_tune = hyperparameter_tuning(
        sequences, targets, word_refs, phn_refs,
        vocab, id2phn, actual_dim,
        train_ids, val_ids, FIXED_CONFIG
    )

    # ── TAHAP 3 ──────────────────────────────────────────────
    print("\n" + "="*70)
    print("  TAHAP 3 — Training Penuh dengan Config Terbaik")
    print("="*70)
    for k in ["n_layers","hidden_dim","learning_rate","dropout",
              "context_type","contexts"]:
        print(f"  {k:<16}: {best_params[k]}")
    print(f"  max_epochs    : {FIXED_CONFIG['epochs_full']}")

    np.random.seed(FIXED_CONFIG["random_seed"])
    best_model = TDNNModel(
        input_dim  = actual_dim,
        hidden_dim = best_params["hidden_dim"],
        contexts   = best_params["contexts"],
        output_dim = len(vocab),
        dropout    = best_params["dropout"],
        lr         = FIXED_CONFIG["lr_warmup"],
    )

    _, history = run_training(
        best_model, train_ids, val_ids, sequences, targets,
        FIXED_CONFIG,
        lr_init    = best_params["learning_rate"],
        max_epochs = FIXED_CONFIG["epochs_full"],
        early_stop = FIXED_CONFIG["early_stop_full"],
        model_save_path = MODEL_BEST_PATH,
        verbose    = True,
    )
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  HasilTraining_TDNN.csv tersimpan")

    if os.path.exists(MODEL_BEST_PATH):
        best_model.load(MODEL_BEST_PATH)

    # ── TAHAP 4 ──────────────────────────────────────────────
    print("\n" + "="*70)
    print("  TAHAP 4 — Evaluasi Model Terbaik")
    print("="*70)

    all_wer_rows = []
    summ_test    = None
    for split_name, split_ids in [("val", val_ids), ("test", test_ids)]:
        wr_rows, summ_wer = evaluate_wer(
            best_model, split_ids, sequences, word_refs, phn_refs,
            id2phn, beam_width=FIXED_CONFIG["beam_width"]
        )
        phn_acc, phn_per = compute_phoneme_accuracy(
            best_model, split_ids, sequences, phn_refs,
            id2phn, beam_width=FIXED_CONFIG["beam_width"]
        )
        print_eval_stats(summ_wer, phn_acc, phn_per, split_name)
        all_wer_rows.extend(wr_rows)
        if split_name == "test": summ_test = summ_wer

    df_w = pd.DataFrame(all_wer_rows)
    df_w[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_w[["audio_id","referensi","hipotesis","WER"]].to_csv(
        OUT_DEC_CSV, index=False)

    print(f"\n  File tersimpan:")
    for p in [OUT_TUNING_CSV, OUT_WER_CSV, OUT_DEC_CSV,
              OUT_HISTORY_CSV, MODEL_BEST_PATH]:
        print(f"    {p}")

    return best_model, summ_test, history, df_tune


if __name__ == "__main__":
    model, summary, history, tuning_results = main()

  TAHAP 1 — Memuat Data & Ekstraksi MFCC
  Folder audio : D:\doc\2026\FINAL_CODE\data\sampled_small
  CSV          : D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv
  Output       : D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/TDNN/out
  Membaca D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv ...
  Shape CSV : (1749, 4)
  Vocab fonem : 31 fonem + 1 blank = 32 unit
  Fonem       : ['<sp>', 'DH', 'NG', 'NY', 'TH', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
  Contoh G2P  : 'hermes punika ugi dados utusanipun dewa ' → ['h', 'e', 'r', 'm', 'e', 's', '<sp>', 'p', 'u', 'n']
  Mengekstrak MFCC dari 1749 file .wav ...
    [200/1749] diproses, 0 dilewati
    [400/1749] diproses, 0 dilewati
    [600/1749] diproses, 0 dilewati
    [800/1749] diproses, 0

KeyboardInterrupt: 